In [0]:
import pyspark.pipelines as dp
from pyspark.sql.functions import from_json, col, to_timestamp
from pyspark.sql.types import StructType, StringType, BooleanType, LongType, TimestampType

In [0]:
schema = (
    StructType()
    .add("title", StringType())
    .add("user", StringType())
    .add("bot", BooleanType())
    .add("wiki", StringType())
    .add("server_name", StringType())
    .add("type", StringType())
    .add("timestamp", LongType())        # epoch seconds from source
    .add("id", LongType())               # natural key for dedup
)


In [0]:
@dp.table(
    name="streaming_demo.silver.wiki_events_slv",
    comment="Cleaned, deduplicated, and validated Wikipedia edit events"
)
@dp.expect_or_drop("valid_id", "id IS NOT NULL")                      # DQ check during load
@dp.expect_or_drop("valid_timestamp", "event_ts IS NOT NULL")         # DQ check during load
@dp.expect("valid_wiki", "wiki IS NOT NULL")                          # DQ metric logging
def silver_wiki_events():
    return (
        dp.read_stream("streaming_demo.bronze.wiki_events_brz")
        .select(from_json(col("raw_json"), schema).alias("data"), "ingest_ts")
        .select("data.*", "ingest_ts")
        .withColumn("event_ts", to_timestamp(col("timestamp")))
        .withWatermark("event_ts", "2 minutes")
        .dropDuplicatesWithinWatermark(["id"])
    )